Generate Collect Earth Online plots for each SIDS country, following the CEO example plot GeoJSON.
Each plot is one cell of the 30m grid for the country's region from `get_gridspec`, so plots are exactly the GeoMAD and
LULC pixels. No separate samples are needed.

1. Load GADM for the country and buffer 200m (antimeridian fixed).
2. Get the 30m grid cells inside the buffered country.
3. Randomly choose the number of plots from those cells.
4. Write a GeoJSON per country in EPSG:4326 with a square polygon per plot, with PLOTID and ExtraInfo (the country name).
5. Also write it as a zipped shapefile, because the CEO UI has a bug preventing GeoJSON input.

In [48]:
from pathlib import Path

import geopandas as gpd
import numpy as np
from odc.geo.geom import Geometry
from odc.geo.xr import rasterize

from ldn.grids import get_gridspec
from ldn.training_data import get_buffered_country
from ldn.utils import DEP_COUNTRIES_AND_CODES, WGS84, get_analysis_epsg

In [49]:
# countries = dict(list(ALL_COUNTRIES.items()))
countries = {"Fiji": "FJI", "Anguilla": "AIA", "Singapore": "SGP"}  # Testing
BUFFER_M = 200
N_PLOTS = 300
# Plot is PLOT_PIXELS x PLOT_PIXELS pixels (1 is a single grid cell). Must divide the 3200 pixel tile size.
PLOT_PIXELS = 1
SEED = 0
PLOTS_DIR = Path("country_plots")
PLOTS_DIR.mkdir(exist_ok=True)

In [ ]:
def make_country_plots(name: str, code: str) -> gpd.GeoDataFrame:
    """Return randomly chosen square plots of grid cells, aligned to the region grid, fully inside a buffered country.

    Args:
        name: Country name.
        code: GADM country code.

    Returns:
        GeoDataFrame of square polygons in EPSG:4326 with PLOTID (string) and ExtraInfo (country name),
        at most N_PLOTS rows.
    """
    region = "pacific" if name in DEP_COUNTRIES_AND_CODES else "non-pacific"
    analysis_crs = get_analysis_epsg(region)
    aoi = get_buffered_country({name: code}, analysis_crs, BUFFER_M).dissolve()
    aoi_geom = Geometry(aoi.to_crs(analysis_crs).geometry.iloc[0], crs=analysis_crs)

    n = PLOT_PIXELS
    xs, ys = [], []
    for _, geobox in get_gridspec(region).tiles_from_geopolygon(aoi_geom):
        inside = rasterize(aoi_geom, geobox).values
        ny, nx = inside.shape
        full_blocks = inside.reshape(ny // n, n, nx // n, n).all(axis=(1, 3))
        by, bx = np.nonzero(full_blocks)
        # Block centre is the mean of its pixel centres
        xs.append(geobox.coords["x"].values.reshape(nx // n, n).mean(axis=1)[bx])
        ys.append(geobox.coords["y"].values.reshape(ny // n, n).mean(axis=1)[by])
    xs, ys = np.concatenate(xs), np.concatenate(ys)

    # TODO: Instead of random, select plots based on WC class so it is stratified?
    keep = np.random.default_rng(SEED).choice(len(xs), size=min(N_PLOTS, len(xs)), replace=False)
    centres = gpd.GeoSeries(gpd.points_from_xy(xs[keep], ys[keep]), crs=analysis_crs)
    half_width = n * get_gridspec(region).resolution.x / 2
    print(f"{name}: {len(xs)} full blocks in country, chose {len(keep)} plots")

    return gpd.GeoDataFrame(
        {"PLOTID": np.arange(1, len(keep) + 1).astype(str), "ExtraInfo": name},
        geometry=centres.buffer(half_width, cap_style="square").to_crs(WGS84),
    )

In [52]:
for name, code in countries.items():
    plots = make_country_plots(name, code)
    # plots.to_file(PLOTS_DIR / f"{name}.geojson", driver="GeoJSON")
    plots.to_file(PLOTS_DIR / f"{name}.shp.zip", driver="ESRI Shapefile")

/Users/wj/Projects/ldn-lulc/ldn-lulc/.venv/lib/python3.14/site-packages/antimeridian/_implementation.py:475: FixWindingWarning: The exterior ring of this shape is wound clockwise. Since this is a common error in real-world geometries, this package is reversing the exterior coordinates of the input shape before running its algorithm. If you know that your input shape is correct (i.e. if your data encompasses both poles), pass `fix_winding=False`. To silence this warning while keeping the fix, pass `fix_winding=True`.
  FixWindingWarning.warn()
/Users/wj/Projects/ldn-lulc/ldn-lulc/.venv/lib/python3.14/site-packages/antimeridian/_implementation.py:734: FixWindingWarning: The exterior ring of this shape is wound clockwise. Since this is a common error in real-world geometries, this package is reversing the exterior coordinates of the input shape before running its algorithm. If you know that your input shape is correct (i.e. if your data encompasses both poles), pass `fix_winding=False`. T

Fiji: 24235599 full blocks in country, chose 300 plots
Anguilla: 106169 full blocks in country, chose 300 plots


/Users/wj/Projects/ldn-lulc/ldn-lulc/.venv/lib/python3.14/site-packages/antimeridian/_implementation.py:475: FixWindingWarning: The exterior ring of this shape is wound clockwise. Since this is a common error in real-world geometries, this package is reversing the exterior coordinates of the input shape before running its algorithm. If you know that your input shape is correct (i.e. if your data encompasses both poles), pass `fix_winding=False`. To silence this warning while keeping the fix, pass `fix_winding=True`.
  FixWindingWarning.warn()
/Users/wj/Projects/ldn-lulc/ldn-lulc/.venv/lib/python3.14/site-packages/antimeridian/_implementation.py:475: FixWindingWarning: The exterior ring of this shape is wound clockwise. Since this is a common error in real-world geometries, this package is reversing the exterior coordinates of the input shape before running its algorithm. If you know that your input shape is correct (i.e. if your data encompasses both poles), pass `fix_winding=False`. T

Singapore: 873425 full blocks in country, chose 300 plots


In [53]:
# Combine into 1 file (renaming PLOTID)
from pathlib import Path

import geopandas as gpd
import pandas as pd

In [54]:
PLOTS_DIR = Path("country_plots")
# COMBINED_PATH = Path("country_plots_combined.geojson")
COMBINED_SHP_ZIP_PATH = Path("country_plots_combined.shp.zip")

combined = gpd.GeoDataFrame(
    pd.concat([gpd.read_file(path) for path in sorted(PLOTS_DIR.glob("*.shp.zip"))], ignore_index=True),
    crs=WGS84,
)
combined["PLOTID"] = (combined.index + 1).astype(str)

# combined.to_file(COMBINED_PATH, driver="GeoJSON")
combined.to_file(COMBINED_SHP_ZIP_PATH, driver="ESRI Shapefile")
print(f"Wrote {len(combined)} plots from {combined['ExtraInfo'].nunique()} countries to {COMBINED_SHP_ZIP_PATH}")
combined.groupby("ExtraInfo").size()

Wrote 900 plots from 3 countries to country_plots_combined.shp.zip


ExtraInfo
Anguilla     300
Fiji         300
Singapore    300
dtype: int64